In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

In [6]:
sentences = [
    "this movie is good",
    "i like this film",
    "this movie is bad",
    "i hate this film"
]
labels = [1, 1, 0, 0]

In [7]:
vocab = {}
for sentence in sentences:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = len(vocab) + 1
vocab_size = len(vocab) + 1  # +1 for padding index

def encode(sentence):
    return [vocab[word] for word in sentence.split()]

encoded_sentences = [encode(s) for s in sentences]

# Pad sequences to same length
max_len = max(len(s) for s in encoded_sentences)
padded_sentences = [s + [0]*(max_len - len(s)) for s in encoded_sentences]

In [8]:
X = torch.tensor(padded_sentences, dtype=torch.long)  # (batch, seq_len)
y = torch.tensor(labels, dtype=torch.long)            # (batch,)

In [9]:
class TextRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size):
        super(TextRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.embedding(x)              # (batch, seq_len, embed_size)
        out, _ = self.rnn(x)               # (batch, seq_len, hidden_size)
        out = self.fc(out[:, -1, :])       # last time step
        return out

model = TextRNN(vocab_size=vocab_size, embed_size=10, hidden_size=16, output_size=2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [10]:
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# Prediction
def predict(sentence):
    encoded = encode(sentence)
    padded = encoded + [0]*(max_len - len(encoded))  # pad to same length
    encoded_tensor = torch.tensor([padded], dtype=torch.long)
    output = model(encoded_tensor)
    prediction = torch.argmax(output, dim=1).item()
    return "Positive" if prediction == 1 else "Negative"

Epoch [20/100], Loss: 0.0126
Epoch [40/100], Loss: 0.0010
Epoch [60/100], Loss: 0.0005
Epoch [80/100], Loss: 0.0004
Epoch [100/100], Loss: 0.0003


In [11]:
print("\nPredictions:")
print("this movie is good ->", predict("this movie is good"))
print("this movie is bad  ->", predict("this movie is bad"))
print("i like this film  ->", predict("i like this film"))
print("i hate this film  ->", predict("i hate this film"))


Predictions:
this movie is good -> Positive
this movie is bad  -> Negative
i like this film  -> Positive
i hate this film  -> Negative
